# 22f - Ablation Study: LSTM on WLASL-1200

**Ablation Study** — identical training setup as Notebook 17h, but replacing
the GRU recurrent layer with an LSTM.

NUM_CLASSES=1200 ≥ 800 → epochs=250, patience=50 (auto).

## Architecture: PoseClassifierLSTM
- Input projection: Linear → LayerNorm → ReLU → Dropout
- **LSTM** (bidirectional, 2 layers, hidden=256) — replaces GRU
- Temporal Attention
- Classifier head

In [ ]:
# === Imports and Configuration ===
# ⬇️  Only this line changes between notebooks 22a / 22b / 22c / ...
SUBSET_PATH = "/home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/subsets/wlasl_1200_subset.json"
# ════════════════════════════════════════════════════════════════

import numpy as np, json, random, time
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import Dict, List, Optional, Tuple
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SUBSET_PATH=Path(SUBSET_PATH)
LANDMARKS_ROOT=Path("/home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl2000")
EXPERIMENTS_BASE=Path("/home/fransando/Dev/tfg-sign-language-recognition/05_experiments")

with open(SUBSET_PATH) as f: subset_meta=json.load(f)
EXPERIMENT_NAME=subset_meta['name']; NUM_CLASSES=subset_meta['num_classes']
OUTPUT_DIR=EXPERIMENTS_BASE/f"{EXPERIMENT_NAME}_lstm"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

if NUM_CLASSES>=800: _epochs=250; _patience=50
else: _epochs=150; _patience=30

CONFIG={'max_frames':64,'input_size':63,'num_classes':NUM_CLASSES,'hidden_size':256,'num_layers':2,
        'dropout':0.4,'bidirectional':True,'batch_size':32,'epochs':_epochs,'lr':1e-3,
        'weight_decay':5e-4,'label_smoothing':0.1,'patience':_patience}

print(f'Date:{datetime.now().strftime("%Y-%m-%d %H:%M")} | Device:{device} | Subset:{EXPERIMENT_NAME}')
print(f'Classes:{NUM_CLASSES} | Epochs:{CONFIG["epochs"]} | Patience:{CONFIG["patience"]}')
print(f'Output: {OUTPUT_DIR}')
stats=subset_meta['statistics']
print(f'Train:{stats["train"]} Val:{stats["val"]} Test:{stats["test"]} MeanV/C:{stats["mean_videos_per_class"]}')

In [ ]:
# === Dataset and DataLoaders ===
class SkeletalAugmentation:
    def __init__(self,rotation=10,scale=(0.9,1.1),translation=0.1,noise=0.01,p=0.5):
        self.r=rotation;self.s=scale;self.t=translation;self.n=noise;self.p=p
    def __call__(self,lm):
        lm=lm.copy()
        if random.random()<self.p:
            a=np.radians(random.uniform(-self.r,self.r));ca,sa=np.cos(a),np.sin(a)
            x,y=lm[:,:,0].copy(),lm[:,:,1].copy();lm[:,:,0]=x*ca-y*sa;lm[:,:,1]=x*sa+y*ca
        if random.random()<self.p: lm*=random.uniform(*self.s)
        if random.random()<self.p:
            lm[:,:,0]+=random.uniform(-self.t,self.t);lm[:,:,1]+=random.uniform(-self.t,self.t)
        if random.random()<self.p: lm+=np.random.normal(0,self.n,lm.shape)
        return lm

class SubsetDataset(Dataset):
    def __init__(self,samples,root,mf=64,aug=None): self.s=samples;self.root=root;self.mf=mf;self.aug=aug
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        it=self.s[i];lm=np.load(self.root/it['path']);T=lm.shape[0]
        if T>self.mf: lm=lm[np.linspace(0,T-1,self.mf,dtype=int)]
        elif T<self.mf: lm=np.concatenate([lm,np.zeros((self.mf-T,21,3))],axis=0)
        if self.aug: lm=self.aug(lm)
        return torch.tensor(lm.reshape(self.mf,-1),dtype=torch.float32),it['gloss_idx']

aug=SkeletalAugmentation()
train_ds=SubsetDataset(subset_meta['samples']['train'],LANDMARKS_ROOT,CONFIG['max_frames'],aug)
val_ds  =SubsetDataset(subset_meta['samples']['val'],  LANDMARKS_ROOT,CONFIG['max_frames'])
test_ds =SubsetDataset(subset_meta['samples']['test'], LANDMARKS_ROOT,CONFIG['max_frames'])
train_loader=DataLoader(train_ds,batch_size=CONFIG['batch_size'],shuffle=True, num_workers=0,pin_memory=True)
val_loader  =DataLoader(val_ds,  batch_size=CONFIG['batch_size'],shuffle=False,num_workers=0)
test_loader =DataLoader(test_ds, batch_size=CONFIG['batch_size'],shuffle=False,num_workers=0)
print(f'Train:{len(train_ds)} Val:{len(val_ds)} Test:{len(test_ds)}')

In [ ]:
# === Model: PoseClassifierLSTM ===
class TemporalAttention(nn.Module):
    def __init__(self,hs):
        super().__init__()
        self.a=nn.Sequential(nn.Linear(hs,hs//2),nn.Tanh(),nn.Linear(hs//2,1,bias=False))
    def forward(self,ro):
        w=F.softmax(self.a(ro).squeeze(-1),dim=1)
        return torch.bmm(w.unsqueeze(1),ro).squeeze(1),w

class PoseClassifierLSTM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        hs=cfg['hidden_size'];bidir=cfg.get('bidirectional',True);rhs=hs*(2 if bidir else 1)
        self.ip=nn.Sequential(nn.Linear(cfg['input_size'],hs),nn.LayerNorm(hs),nn.ReLU(),nn.Dropout(cfg['dropout']))
        self.lstm=nn.LSTM(input_size=hs,hidden_size=hs,num_layers=cfg['num_layers'],batch_first=True,
                          dropout=cfg['dropout'] if cfg['num_layers']>1 else 0,bidirectional=bidir)
        self.attn=TemporalAttention(rhs)
        self.cls=nn.Sequential(nn.Dropout(cfg['dropout']),nn.Linear(rhs,hs),nn.ReLU(),
                                nn.Dropout(cfg['dropout']),nn.Linear(hs,cfg['num_classes']))
    def forward(self,x,return_attention=False):
        x=self.ip(x)
        rnn_out,_=self.lstm(x)  # _ discards (h_n, c_n)
        ctx,attn_w=self.attn(rnn_out)
        logits=self.cls(ctx)
        return (logits,attn_w) if return_attention else logits

model=PoseClassifierLSTM(CONFIG).to(device)
print(f'PoseClassifierLSTM | params={sum(p.numel() for p in model.parameters()):,} | classes={NUM_CLASSES}')

In [ ]:
# === Training and Evaluation Functions ===
def train_epoch(model,loader,crit,opt):
    model.train();tl,c,tot=0.0,0,0
    for x,y in loader:
        x,y=x.to(device),y.to(device);opt.zero_grad()
        o=model(x);l=crit(o,y);l.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
        tl+=l.item()*x.size(0);c+=o.argmax(1).eq(y).sum().item();tot+=x.size(0)
    return tl/tot,100*c/tot

@torch.no_grad()
def evaluate(model,loader,top_k=(1,5)):
    model.eval();tot=0;tc={k:0 for k in top_k};ap,al=[],[]
    for x,y in loader:
        x,y=x.to(device),y.to(device);lg=model(x);tot+=y.size(0)
        _,pk=lg.topk(max(top_k),1,True,True)
        for k in top_k: tc[k]+=pk[:,:k].eq(y.view(-1,1).expand_as(pk[:,:k])).any(dim=1).sum().item()
        ap.extend(lg.argmax(1).cpu().numpy());al.extend(y.cpu().numpy())
    return {f'top{k}_acc':100.0*tc[k]/tot for k in top_k},np.array(ap),np.array(al)
print('Functions defined.')

In [ ]:
# === Training Loop ===
crit=nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
opt=AdamW(model.parameters(),lr=CONFIG['lr'],weight_decay=CONFIG['weight_decay'])
sched=CosineAnnealingLR(opt,T_max=CONFIG['epochs'])
hist={'train_loss':[],'train_acc':[],'val_top1':[],'val_top5':[]}
bvt1=0.0;be=0;pc=0;ts=time.time();CP=OUTPUT_DIR/'best_model.pt'
print(f'Training {EXPERIMENT_NAME}_lstm | epochs={CONFIG["epochs"]} patience={CONFIG["patience"]}'); print('='*65)
for epoch in range(CONFIG['epochs']):
    t0=time.time();tl,ta=train_epoch(model,train_loader,crit,opt)
    vm,_,_=evaluate(model,val_loader);sched.step()
    hist['train_loss'].append(tl);hist['train_acc'].append(ta)
    hist['val_top1'].append(vm['top1_acc']);hist['val_top5'].append(vm['top5_acc'])
    if vm['top1_acc']>bvt1:
        bvt1=vm['top1_acc'];be=epoch+1;pc=0
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'val_top1':bvt1,
                    'config':CONFIG,'subset':EXPERIMENT_NAME,'architecture':'PoseClassifierLSTM'},CP)
    else: pc+=1
    if (epoch+1)%10==0 or epoch==0:
        print(f'Ep{epoch+1:4d} L{tl:.3f} Tr{ta:.1f}% V1{vm["top1_acc"]:.1f}% V5{vm["top5_acc"]:.1f}% Best{bvt1:.1f}% {time.time()-t0:.1f}s')
    if pc>=CONFIG['patience']: print(f'Early stop ep{epoch+1}'); break
elapsed=time.time()-ts; best_val_top1=bvt1; best_epoch=be; history=hist
print(f'Done {elapsed/60:.1f}min | Best Val Top-1:{best_val_top1:.1f}% ep{best_epoch}')

In [ ]:
# === Training Curves ===
fig,axes=plt.subplots(1,3,figsize=(16,4));er=range(1,len(history['train_loss'])+1)
axes[0].plot(er,history['train_loss'],color='steelblue');axes[0].set_title('Loss');axes[0].grid(True,alpha=0.3)
axes[1].plot(er,history['train_acc'],label='Train',color='steelblue')
axes[1].plot(er,history['val_top1'],label='Val Top-1',color='darkorange')
axes[1].axhline(best_val_top1,color='red',linestyle='--',label=f'Best:{best_val_top1:.1f}%')
axes[1].set_title('Accuracy');axes[1].legend();axes[1].grid(True,alpha=0.3)
axes[2].plot(er,history['val_top5'],color='green',label='Top-5')
axes[2].plot(er,history['val_top1'],color='darkorange',label='Top-1')
axes[2].set_title('Val Top-1 vs Top-5');axes[2].legend();axes[2].grid(True,alpha=0.3)
plt.suptitle(f'{EXPERIMENT_NAME}_lstm — PoseClassifierLSTM',fontsize=13)
plt.tight_layout();plt.savefig(OUTPUT_DIR/'training_curves.png',dpi=150,bbox_inches='tight');plt.show()

In [ ]:
# === Final Evaluation ===
ckpt=torch.load(OUTPUT_DIR/'best_model.pt',map_location=device,weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
val_metrics,val_preds,val_labels=evaluate(model,val_loader)
test_metrics,test_preds,test_labels=evaluate(model,test_loader)
print(f'Val  Top-1:{val_metrics["top1_acc"]:.1f}% Top-5:{val_metrics["top5_acc"]:.1f}% (n={len(val_ds)})')
print(f'Test Top-1:{test_metrics["top1_acc"]:.1f}% Top-5:{test_metrics["top5_acc"]:.1f}% (n={len(test_ds)})')

In [ ]:
# === Per-Class + Confused Pairs + Save JSON ===
idx_to_gloss=subset_meta['idx_to_gloss']
per_class_acc={}
for ci in range(NUM_CLASSES):
    m=test_labels==ci
    if m.sum()==0: continue
    per_class_acc[ci]={'gloss':idx_to_gloss[str(ci)],'acc':float(100.0*(test_preds[m]==ci).sum()/m.sum()),'n_test':int(m.sum())}
accs=[v['acc'] for v in per_class_acc.values()]
print(f'Mean:{np.mean(accs):.1f}% Std:{np.std(accs):.1f}% 100%:{sum(1 for a in accs if a==100)} 0%:{sum(1 for a in accs if a==0)}')
wm=test_preds!=test_labels; pc2=Counter(zip(test_labels[wm].tolist(),test_preds[wm].tolist()))
print('Top-20 confused:')
for (t,p),c in pc2.most_common(20): print(f'  {idx_to_gloss.get(str(t),str(t)):<22}→{idx_to_gloss.get(str(p),str(p)):<22}({c}x)')

results={'experiment':f'{EXPERIMENT_NAME}_lstm','architecture':'PoseClassifierLSTM',
         'date':datetime.now().strftime('%Y-%m-%d %H:%M'),'subset_path':str(SUBSET_PATH),
         'num_classes':NUM_CLASSES,'config':CONFIG,
         'dataset':{'train':len(train_ds),'val':len(val_ds),'test':len(test_ds)},
         'training':{'best_val_top1':best_val_top1,'best_epoch':best_epoch,
                     'total_epochs':len(history['train_acc']),'training_time_s':round(elapsed,1)},
         'val_metrics':val_metrics,'test_metrics':test_metrics,
         'per_class_accuracy':{str(k):v for k,v in per_class_acc.items()},
         'history':{k:[round(float(v),4) for v in vals] for k,vals in history.items()}}
with open(OUTPUT_DIR/'results.json','w') as f: json.dump(results,f,indent=2,ensure_ascii=False)
print(f'\nSaved: {OUTPUT_DIR}/results.json')
for p in sorted(OUTPUT_DIR.iterdir()): print(f'  {p.name} ({p.stat().st_size//1024}KB)')